In [1]:
import os

In [2]:
os.chdir("../")

#### Update the Entitiy

In [3]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir: Path
    data_path: Path
    tokenizer_name: Path

#### Update Configuration Manager

In [4]:
from textSummarizer.constants import *
from textSummarizer.utils.common import read_yaml, create_directories

In [5]:
class ConfigurationManager:
    def __init__(self, config_file_path=CONFIG_FILE_PATH, params_file_path=PARAMS_FILE_PATH):
        self.config = read_yaml(config_file_path)
        self.params = read_yaml(params_file_path)
    
    def get_data_transformation_config(self) -> DataTransformationConfig:
        data_transformation_config = self.config.data_transformation

        create_directories([data_transformation_config.root_dir])

        data_transformation_config = DataTransformationConfig(
            root_dir = data_transformation_config.root_dir,
            data_path = data_transformation_config.data_path,
            tokenizer_name = data_transformation_config.tokenizer_name
        )
        return data_transformation_config

#### Update the components

In [6]:
import os
from textSummarizer.logging import logger
from transformers import AutoTokenizer
from datasets import load_from_disk

[2026-02-25 11:20:25,338: INFO: config]: TensorFlow version 2.20.0 available.


In [7]:
class DataTransformation:
    
    def __init__(self, config : DataTransformationConfig):
        self.config = config
        self.tokenizer = AutoTokenizer.from_pretrained(self.config.tokenizer_name)

    def convert_examples_to_features(self, example_batch):

        inputs = self.tokenizer(example_batch["dialogue"], max_length=1024, truncation=True)

        labels = self.tokenizer(example_batch["summary"], max_length=128, truncation=True)

        return {
            "input_ids": inputs["input_ids"],
            "attention_mask": inputs["attention_mask"],
            "labels": labels["input_ids"],
        }
    
    def convert(self):
        logger.info("Loading dataset from disk.")
        dataset = load_from_disk(self.config.data_path)

        logger.info("Converting examples to features.")
        dataset_pt = dataset.map(self.convert_examples_to_features, batched=True)

        logger.info("Saving transformed dataset to disk.")
        dataset_pt.save_to_disk(os.path.join(self.config.root_dir, "transformed_dataset"))

#### Update the Pipeline

In [8]:
try:
    config = ConfigurationManager()
    data_transformation_config = config.get_data_transformation_config()
    data_transformation = DataTransformation(config=data_transformation_config)
    data_transformation.convert()
except Exception as e:
    raise e

[2026-02-25 11:20:27,312: INFO: _client]: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
[2026-02-25 11:20:27,379: INFO: _client]: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/config.json "HTTP/1.1 200 OK"
[2026-02-25 11:20:27,692: INFO: _client]: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
[2026-02-25 11:20:27,756: INFO: _client]: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/tokenizer_config.json "HTTP/1.1 200 OK"
[2026-02-25 11:20:28,058: INFO: _client]: HTTP Request: GET https://huggingface.co/api/models/google/pegasus-cnn_dailymail/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
[2026

Map:   0%|          | 0/14732 [00:00<?, ? examples/s]

Map:   0%|          | 0/819 [00:00<?, ? examples/s]

Map:   0%|          | 0/818 [00:00<?, ? examples/s]

[2026-02-25 11:20:32,525: INFO: 2640092418]: Saving transformed dataset to disk.


Saving the dataset (0/1 shards):   0%|          | 0/14732 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/819 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/818 [00:00<?, ? examples/s]